In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import squidpy as sq
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import hdbscan
from scipy.cluster.hierarchy import leaves_list
import sys
sys.path.append('../../scripts')
import coda
import readwrite
cfg = readwrite.config()

def concat_samples(ads, correction_method, segmentation, condition=None, panel=None):
    ads_corr_method = ads[correction_method]

    def is_match(key):
        return (
            segmentation in key and
            (condition is None or condition in key) and
            (panel is None or panel in key)
        )

    adata = sc.concat({k: v for k, v in ads_corr_method.items() if is_match(k)},label='dataset_id', join='outer')
    adata.obs[xenium_levels] = pd.DataFrame(adata.obs['dataset_id'].tolist(),index=adata.obs.index,columns=xenium_levels)
    return adata

/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/anndata/__init__.py:44: F

### Read all xenium samples

In [3]:
# input params
correction_method = 'raw'
segmentation = 'proseg_expected'
condition = 'CRC'
panel = 'all'

xenium_dir = Path(cfg['xenium_processed_dir'])
xenium_count_correction_dir = Path(cfg['xenium_count_correction_dir'])
xenium_std_seurat_analysis_dir = Path(cfg['xenium_std_seurat_analysis_dir'])
xenium_cell_type_annotation_dir = Path(cfg['xenium_cell_type_annotation_dir'])
results_dir = Path(cfg['results_dir'])

xenium_levels = ['segmentation','condition','panel','donor','sample']
normalisation = 'lognorm'
reference = 'GEO_GSE178341' # 'GEO_GSE178341' GEO_GSE236581
method = 'rctd_class_aware'
level = 'Level1'

donor_palette = pd.read_csv(cfg['xenium_metadata_dir'] + 'donor_palette.csv',index_col=0)
gene_sets_palette = pd.read_csv(cfg['xenium_metadata_dir'] + 'gene_sets_palette.csv',index_col=0)
cell_type_palette = pd.read_csv(cfg['xenium_metadata_dir'] + 'cell_type_palette.csv',index_col=0)

# fixed params
BATCH_KEY = 'dataset_id'
SPATIAL_KEY = 'spatial'
N_CLUSTERS_RANGE = (5,19)
MAX_RUNS = 10
CONVERGENCE_TOL = 0.001
OUTPUT_LABELS = results_dir/'xenium/cellcharter/labels.parquet'
OUTPUT_SCVI_MODEL = results_dir/'xenium/cellcharter/scvi_model'
OUTPUT_CELLCHARTER_MODELS = results_dir/'xenium/cellcharter/cellcharter_models'
OUTPUT_PLOT = results_dir/'xenium/cellcharter/autok_stability.png'



# read samples
xenium_paths, xenium_annot_paths = readwrite.discover_xenium_paths(
    analysis_dir=xenium_std_seurat_analysis_dir,
    data_dir=xenium_dir,
    annotation_dir=xenium_cell_type_annotation_dir,
    correction_dir=xenium_count_correction_dir,
    normalisation=normalisation,
    reference=reference,
    method=method,
    level=level,
    correction_methods_filter=[correction_method],
    segmentations_filter=[segmentation],
    conditions_filter=[condition] if condition != 'all' else None,
    panels_filter=[panel] if panel != 'all' else None
)


# set transcripts=True to load individual transcripts positions)
if correction_method != 'raw':
    ads = readwrite.read_count_correction_samples(xenium_paths,[correction_method])
else:
    ads = {}
    ads['raw'] = readwrite.read_xenium_samples(xenium_paths['raw'], anndata=True, pool_mode="thread", max_workers=6)

# add cell type annotation from raw to all correction methods
readwrite.read_annotations(ads, [correction_method], xenium_annot_paths, level, max_workers=8)

# concat samples
adata = sc.concat({k: v for k, v in ads[correction_method].items() },label='dataset_id', join='outer')
adata.obs[xenium_levels] = pd.DataFrame(adata.obs['dataset_id'].tolist(),index=adata.obs.index,columns=xenium_levels)
adata.obs['correction_method'] = correction_method

# apply QC
sc.pp.calculate_qc_metrics(adata, inplace=True, percent_top=False)
idx_qc = (adata.obs["total_counts"] > 20)
print('Filtering out',sum(~idx_qc),'cells with less than 20 counts')
adata = adata[idx_qc].copy()

# info about the loaded cohort
df_cohort_info = pd.DataFrame([k+(corr_method,) for corr_method in [correction_method] for k in ads[corr_method].keys() ], columns=xenium_levels+['correction_method'])

/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/spatialdat

Filtering out 48083 cells with less than 20 counts


/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/anndata/_core/anndata.py:1774: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


## overall composition

In [ ]:
df_props = (
    adata.obs
    .groupby(xenium_levels)['Level1']
    .value_counts(normalize=True)
    .unstack()
    .reset_index()
)
df_props['full_sample_id'] = df_props[xenium_levels].astype(str).agg(' | '.join, axis=1)
df_props = df_props.query(f"segmentation == '{segmentation}'")

# Step 2: Dynamically set the figure size
num_samples = len(df_props.index)
fig_height = max(6, num_samples * 0.4) 

# Step 3: Create the figure and axes object
fig, ax = plt.subplots(figsize=(12, fig_height))

# Step 4: Plot directly onto the created axes object
df_props.query(f"segmentation == '{segmentation}'").plot(
    kind='barh',
    stacked=True,
    ax=ax,
    width=0.8
)


# Step 6: Customize the plot
ax.set_title(f'Proportion of RCTD Cell Types by Sample - {segmentation} - {reference} - {level}', fontsize=16)
ax.set_xlabel('Proportion', fontsize=12)
ax.set_ylabel('Sample ID', fontsize=12)
ax.set_yticklabels(df_props['donor'])
ax.legend(title='Level1', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

df_props.to_csv(f'../../scratch/celltype_composition_{segmentation}_{reference}_{level}.csv')

## coda

In [3]:
adata.obs['sample']=adata.obs['sample'].astype('category')
sq.gr.spatial_neighbors(adata, radius=20, spatial_key='spatial',library_key='sample',coord_type='generic',set_diag=True)

knnidx,knndist = coda.sparse_to_knn(adata.obsp['spatial_connectivities'])
adata_ilr = coda.get_ilr(adata, knnidx=knnidx, label_key=level)

/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/anndata/_core/anndata.py:1774: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [ ]:
# adata_ilr.obs['cell_id'] = adata_ilr.obs_names
# adata_ilr.obs['full_id'] = adata_ilr.obs[xenium_levels+['cell_id']].agg('_'.join, axis=1)
# df_comp = pd.DataFrame(adata_ilr.obsm['X_knnlabels'],
#     index=adata_ilr.obs['full_id'],
#     columns=adata_ilr.uns['X_knnlabels_columns'])
# df_comp.to_parquet(f'../../scratch/df_composition_{segmentation}_{reference}_{level}.parquet')

In [ ]:
cols = [level,'condition','donor','sample']

for i in range(2):
    adata_ilr.obs[f'X_ilr_pca{i}'] = adata_ilr.obsm['X_ilr_pca'][:,i]

sc.pl.embedding(adata_ilr,'X_ilr_pca',color=cols,s=5,alpha=.5,ncols=2)
# sc.pl.umap(adata_ilr,color=cols,s=5,alpha=.5,ncols=2)


plt.plot(adata_ilr.uns['ilr_pca']['explained_variance_ratio_'])
plt.ylabel('explained_variance_ratio')
plt.xlabel('component rank')
plt.show()


# sort by PC1 and normalize to 1
PC1 = adata_ilr.obsm['X_ilr_pca'][:,0]
df_plot = pd.DataFrame(adata_ilr.obsm['X_composition'],columns=adata_ilr.uns['X_knnlabels_columns'])
df_plot = df_plot.iloc[np.argsort(PC1)].T
df_plot.columns = np.arange(df_plot.shape[1])


f, axs = plt.subplots(2,1,figsize=(30,10))

ax = axs[0]
ax = sns.heatmap(data=df_plot,ax=ax)
ax.set_xticks([])
ax.set_yticklabels(df_plot.index, rotation = 0)
ax.set_ylabel('Cell types')
ax.set_xlabel('Sorted ILR PC1 index')


ax = axs[1]
ax.stackplot(df_plot.columns, *df_plot.values, labels=df_plot.index)
ax.set_xticks([])
ax.set_yticks([])
ax.set_ylabel('% of neighboring cells')
ax.set_xlabel('ILR PC1')
ax.legend(
    title='Cell Type', 
    loc='center left', 
    bbox_to_anchor=(1.0, 0.5)
)
ax.margins(x=0, y=0)
plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to make space for legend
plt.show()

In [ ]:
sq.gr.nhood_enrichment(adata_ilr, cluster_key=level)
sq.gr.interaction_matrix(adata_ilr, cluster_key=level)
# sq.gr.co_occurrence(adata_ilr, cluster_key=level)
sq.pl.nhood_enrichment(adata_ilr, cluster_key=level, method="average", figsize=(5, 5))
sq.pl.interaction_matrix(adata_ilr, cluster_key=level, method="average", figsize=(5, 5))
# sq.pl.co_occurrence(adata_ilr, cluster_key=level, clusters="B", figsize=(8, 5))

## coda plots per covariate

In [2]:
df_metadata = pd.read_csv(cfg['xenium_metadata_dir'] + 'Metadata_tumors_CRC.csv',index_col=0)
df_metadata['donor'] = df_metadata['Patient_ID'].str.replace('0YRI', 'OYRI').astype(str)
df_metadata_donor_lvl = df_metadata[['CMS','CRIS','MSI','Site','Organ','donor']].drop_duplicates()
adata.obs = adata.obs.join(df_metadata_donor_lvl.set_index('donor'),on='donor')

adata.obs['sample']=adata.obs['sample'].astype('category')
sq.gr.spatial_neighbors(adata, radius=20, spatial_key='spatial',library_key='sample',coord_type='generic',set_diag=True)
knnidx,knndist = coda.sparse_to_knn(adata.obsp['spatial_connectivities'])
adata_ilr = coda.get_ilr(adata, knnidx=knnidx, label_key=level)

NameError: name 'adata' is not defined

In [ ]:
np.random.seed(0)

X = pd.DataFrame(adata_ilr.obsm['X_composition'],columns=adata_ilr.uns['X_knnlabels_columns'])
labels = adata_ilr.obs['donor']

# Build HDBSCAN model
clusterer = hdbscan.HDBSCAN(
    gen_min_span_tree=True
).fit(X)
tree = clusterer.single_linkage_tree_
clusters = tree.get_clusters(cut_distance=0)
# soft_labels = hdbscan.membership_vector(clusterer)
linkage_matrix = clusterer.single_linkage_tree_.to_numpy()
order = leaves_list(linkage_matrix)

X_ordered = X.iloc[order]
labels_ordered = labels[order]
row_colors = [donor_palette.loc[l].values[0] for l in labels_ordered]

# Create clustermap with clustering disabled
g = sns.clustermap(
    X_ordered,
    row_cluster=False,     # disable row clustering
    col_cluster=False,     # disable column clustering
    row_colors=row_colors,
    cmap="Purples",
    figsize=(12, 16)
)

g.ax_heatmap.set_yticks([])       # remove ticks
g.ax_heatmap.set_ylabel("")       # remove label
g.ax_row_dendrogram.set_visible(False)  # optional: hide empty space
g.ax_heatmap.tick_params(axis='x', labelsize=12)

p = Path(cfg['figures_dir']+f'xenium/coda/{correction_method}/{segmentation}/CRC_composition_clustermap.png')
p.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(p,bbox_inches='tight',dpi=300)
plt.close()

/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/tmp/ipykernel_1874788/1607910423.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  labels_ordered = labels[order]


In [ ]:
exclude_all_malignant = False

# sort by PC1 across samples
PC1 = adata_ilr.obsm['X_ilr_pca'][:,0]
df_plot = pd.DataFrame(adata_ilr.obsm['X_composition'],columns=adata_ilr.uns['X_knnlabels_columns'])

idx_malignant = np.where(adata_ilr.uns['X_knnlabels_columns']=='Epi')[0]
idx_some_malignant = (adata_ilr.obsm['X_composition'][:,idx_malignant]<1).flatten() & (adata_ilr.obsm['X_composition'][:,idx_malignant]>0).flatten()

# plot
labels_for_legend = None
f, axs = plt.subplots(3,5,figsize=(15,6))
axs = axs.flat
for i, (donor, sample) in enumerate(adata_ilr.obs[['donor','sample']].drop_duplicates().itertuples(index=False)):
    ax = next(axs)

    mask = adata_ilr.obs['sample']==sample
    if exclude_all_malignant:
        mask &= idx_some_malignant

    idx_sample = np.where(mask)[0]
    df_plot_sample = df_plot.iloc[idx_sample].copy()
    df_plot_sample = df_plot_sample.iloc[np.argsort(PC1[idx_sample])].T
    df_plot_sample.columns = np.arange(df_plot_sample.shape[1])

    colors = [cell_type_palette.loc[label,'color'] for label in df_plot_sample.index]

    ax.set_title(donor)
    stack = ax.stackplot(df_plot_sample.columns, *df_plot_sample.values, labels=df_plot_sample.index, colors=colors)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.margins(x=0, y=0)

    if labels_for_legend is None:
        labels_for_legend = [l.get_label() for l in stack]


plt.tight_layout(rect=[0, 0, 0.85, 1])  # leave space on the right

# Add a single global legend
f.legend(
    labels_for_legend,            # labels
    title='Cell Type',            # legend title
    loc='center right',           # position relative to figure
    bbox_to_anchor=(0.95, 0.5),  # fine-tune placement
    fontsize=10
)

p = Path(cfg['figures_dir']+f'xenium/coda/{correction_method}/{segmentation}/CRC_composition_global_ordering_{exclude_all_malignant=}.png')
p.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(p,bbox_inches='tight',dpi=300)
plt.close()

In [ ]:
# same as above but ILR PC1 computed separately for each sample

exclude_all_malignant = False
labels_for_legend = None
f, axs = plt.subplots(3,5,figsize=(15,6))
axs = axs.flat

for i, (donor, sample) in enumerate(adata.obs[['donor','sample']].drop_duplicates().itertuples(index=False)):
    ax = next(axs)

    mask = adata.obs['sample']==sample
    adata_sample = adata[mask].copy()

    sq.gr.spatial_neighbors(adata_sample, radius=20, spatial_key='spatial',coord_type='generic',set_diag=True)
    knnidx,knndist = coda.sparse_to_knn(adata_sample.obsp['spatial_connectivities'])
    adata_ilr_sample = coda.get_ilr(adata_sample, knnidx=knnidx, label_key=level)

    df_plot = pd.DataFrame(adata_ilr_sample.obsm['X_composition'],columns=adata_ilr_sample.uns['X_knnlabels_columns'])

    # sort by PC1
    PC1 = adata_ilr_sample.obsm['X_ilr_pca'][:,0]
    if donor in ['OUC1','169V','1EGQ']:
        PC1 = -PC1
        
    if exclude_all_malignant:
        idx_malignant = np.where(adata_ilr_sample.uns['X_knnlabels_columns']=='Epi')[0]
        idx_some_malignant = np.where(((adata_ilr_sample.obsm['X_composition'][:,idx_malignant]<1).flatten() 
                            & (adata_ilr_sample.obsm['X_composition'][:,idx_malignant]>0).flatten()))
        idx = idx_some_malignant
    else:
        idx = np.arange(adata_ilr_sample.obsm['X_composition'].shape[0])

    df_plot_sample = df_plot.iloc[idx].copy()
    df_plot_sample = df_plot_sample.iloc[np.argsort(PC1[idx])].T
    df_plot_sample.columns = np.arange(df_plot_sample.shape[1])

    colors = [cell_type_palette.loc[label,'color'] for label in df_plot_sample.index]

    ax.set_title(donor)
    stack = ax.stackplot(df_plot_sample.columns, *df_plot_sample.values, labels=df_plot_sample.index, colors=colors)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.margins(x=0, y=0)

    if labels_for_legend is None:
        labels_for_legend = [l.get_label() for l in stack]


plt.tight_layout(rect=[0, 0, 0.85, 1])  # leave space on the right

# Add a single global legend
f.legend(
    labels_for_legend,            # labels
    title='Cell Type',            # legend title
    loc='center right',           # position relative to figure
    bbox_to_anchor=(0.95, 0.5),  # fine-tune placement
    fontsize=10
)

p = Path(cfg['figures_dir']+f'xenium/coda/{correction_method}/{segmentation}/CRC_composition_{exclude_all_malignant=}.png')
p.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(p,bbox_inches='tight',dpi=300)

In [ ]:
# sort by PC1
PC1 = adata_ilr.obsm['X_ilr_pca'][:,0]
df_plot = pd.DataFrame(adata_ilr.obsm['X_composition'],columns=adata_ilr.uns['X_knnlabels_columns'])

for var in ['CMS', 'CRIS', 'MSI', 'Site', 'Organ']:

    u_values = adata_ilr.obs[var].dropna().unique()

    labels_for_legend = None
    f, axs = plt.subplots(1,len(u_values),figsize=(10,3))
    axs = axs.flat
    for i, v in enumerate(u_values):
        ax = next(axs)

        idx_sample = np.where(adata_ilr.obs[var]==v)[0]
        df_plot_sample = df_plot.iloc[idx_sample].copy()
        df_plot_sample = df_plot_sample.iloc[np.argsort(PC1[idx_sample])].T
        df_plot_sample.columns = np.arange(df_plot_sample.shape[1])

        ax.set_title(v)
        stack = ax.stackplot(df_plot_sample.columns, *df_plot_sample.values, labels=df_plot_sample.index)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.margins(x=0, y=0)

        if labels_for_legend is None:
            labels_for_legend = [l.get_label() for l in stack]


    plt.tight_layout(rect=[0, 0, 0.85, 1])  # leave space on the right

    # Add a single global legend
    f.legend(
        labels_for_legend,            # labels
        title='Cell Type',            # legend title
        loc='center right',           # position relative to figure
        bbox_to_anchor=(0.95, 0.5),  # fine-tune placement
        fontsize=10
    )

    plt.show()